In [ ]:
##############################################
##### for development only                   #
# %load_ext autoreload                       #
# %autoreload 2                              #
##############################################

import pyphyschemtools as t4pPC
from pyphyschemtools import fg, hl, bg, color

t4pPC.init('t4pPC')
%matplotlib inline

# get_ppct_data() is a helper function made to easily retrieve absolute paths for built-in example files (XYZ, Excel, etc.)
# that are available in the data_examples/ subfolder as part of the pyphyschemtools library

# This get_ppct_data() utility is specifically designed for accessing built-in example data bundled
# with pyphyschemtools. For your own local files, please use standard Python paths or pathlib

from pyphyschemtools import get_ppct_data

%load_ext autoreload
%autoreload 2

# Module `symmetry`: demo

Overview of the features:
1. symmetry operations as 3×3 matrices
2. character tables (55 point groups, basis functions computed)
3. `MolSym`: a molecule built from a SMILES or an xyz file (read by `molView`); point group detection (pymatgen), standard orientation, symmetrization
4. atom tracking, Γ_atoms, reduction, basis of atomic orbitals
5. 3D visualization (py3Dmol): symmetry elements, before/after an operation
6. vibrations: Γ_3N and Γ_vib with `reduce(chi)`, IR / Raman activity, basis of bond stretchings

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from pyphyschemtools.symmetry import (MolSym, character_table, available_groups, identify, show_matrix, inv,
                                      reduce_representation, ir_raman_activity,
                                      identity, inversion, Cn, sigma, Sn)
from pyphyschemtools.Chem3D import molView
pd.set_option("display.max_columns", 30)

## 1. Symmetry operations = 3×3 matrices

`Cn(n, axis, k)`, `sigma(plane)`, `Sn(n, axis, k)`, `inversion()`, `identity()`.

- axis: `"x"`, `"y"`, `"z"` (default) or any direction, e.g. `(1, 1, 1)`
- plane: `"xy"` (default), `"xz"`, `"yz"` or, for any other plane, its normal vector, e.g. `(1, -1, 0)` for the diagonal plane containing z and the bisector of x and y (σd of CH4, D4h...)

In [ ]:
C3 = Cn(3)                  # C3 about z
C2 = Cn(2)
sigma_h = sigma("xy")       # xy plane
sigma_d = sigma((1, -1, 0)) # diagonal plane: contains z and the direction (1, 1, 0)
S4 = Sn(4)                  # S4 about z
print(np.round(C3, 3))
print("det:", round(np.linalg.det(C3)), round(np.linalg.det(sigma_h)), round(np.linalg.det(S4)))

### Checking a relation

In [ ]:
print("C3³ = E:", np.allclose(C3 @ C3 @ C3, identity()))
print("C3² = C3⁻¹:", np.allclose(C3 @ C3, inv(C3)))
print("C3⁻¹ = transpose of C3 (orthogonal matrix):", np.allclose(inv(C3), C3.T))
print("C3·C3⁻¹ = E:", np.allclose(C3 @ inv(C3), identity()))
print("σh·C2 = i:", np.allclose(sigma_h @ C2, inversion()))
print("S4 = σh·C4:", np.allclose(S4, sigma_h @ Cn(4)))

### Naming the result of a product: `identify`

In [ ]:
print("C3·C3       =", identify(C3 @ C3))
print("C3·C3·C3    =", identify(C3 @ C3 @ C3))
print("σh·C2       =", identify(sigma_h @ C2))
print("C6·C6       =", identify(Cn(6) @ Cn(6)))
print("S4·S4       =", identify(S4 @ S4))
print("S4·S4·S4    =", identify(S4 @ S4 @ S4))
print("S6³         =", identify(Sn(6) @ Sn(6) @ Sn(6)))
print("C3⁻¹        =", identify(inv(C3)))
print("S4⁻¹        =", identify(inv(S4), with_inverse=True))
print("σ(xz)·σ(yz) =", identify(sigma("xz") @ sigma("yz")))
print("C2(x)·C2(y) =", identify(Cn(2, "x") @ Cn(2, "y")))
print("C3(1,1,1)   =", identify(Cn(3, (1, 1, 1))))
print("σd·σ(xz)    =", identify(sigma_d @ sigma("xz")))

`inv(M)`: inverse of an operation (= its transpose, the matrix being orthogonal).

The product of two reflections is a rotation by twice the angle between the planes: σd·σ(xz) = C4(z) (planes at 45°).

### Displaying a matrix: `show_matrix` (exact forms 1/2, √3/2... when possible)

In [ ]:
show_matrix(C3)
show_matrix(C3 @ C3)               # name given by identify
show_matrix(inv(C3))               # inverse: C3⁻¹ = C3²
show_matrix(S4, "S_4")             # imposed name
show_matrix(Cn(3, (1, 1, 1)))      # a permutation matrix
show_matrix(Cn(5), exact=False)    # decimal values

### Non-commutativity: C3·σv ≠ σv·C3 (group C3v)

In [ ]:
sv = sigma("xz")            # σv(xz)
print("C3·σv =", identify(C3 @ sv))
print("σv·C3 =", identify(sv @ C3))

σ(z,φ): vertical plane containing z and the direction at φ from the x axis. The three σv planes of C3v are σ(z,0°) = σ(xz), σ(z,60°) and σ(z,120°):
C3·σv and σv·C3 are two of the other σv planes, but not the same one → the operations do not commute.

## 2. Character tables

In [ ]:
print(available_groups())

In [ ]:
display(character_table("C2v"))
display(character_table("D3h"))

Another name of a group is accepted, with a message: for odd n, Sn contains σh and Cn, it is the group Cnh.

In [ ]:
display(character_table("S3", functions=False))

Pairs of complex conjugate irreducible representations (C3...C8, S4, S6, S8, C3h...C8h, T, Th):
- by default, a single real row (sum of the two characters, e.g. 2, -1, -1 for C3), marked with `*`: it counts as one pair in `reduce()`
- `complex=True`: the two components ¹E and ²E, with complex conjugate characters (ε, ε*). Reduction with the complex characters: n_i = (1/h) Σ g χ(R) χ_i(R)*

In [ ]:
display(character_table("C3"))
display(character_table("C3", complex=True))   # the two components of the pair, ε = exp(2πi/3)
display(character_table("C6", complex=True))
display(character_table("Ih", functions=False))

### Reduction without a molecule: `reduce_representation`, `ir_raman_activity`

Characters given by hand, one per class, in the order of the columns of the character table.

In [ ]:
reduce_representation("C3v", [4, 1, 2])
reduce_representation("Td", [9, 0, 1, -1, 3])
ir_raman_activity("Td", [9, 0, 1, -1, 3])

## 3. Building a molecule: `MolSym.from_molView`

A molecule is a `MolSym` object, built from its **3D structure** (position of each atom). `MolSym` does not read structures itself: it delegates this task to `molView` (Chem3D module), which reads a SMILES (3D structure from PubChem, or generated by RDKit), an xyz file, a PubChem CID...

At its creation, the molecule:
- gets its point group, detected by pymatgen (or given with `group=`), and stored in the attribute `group`;
- is placed in the standard orientation of its point group (Mulliken conventions: molecular plane yz for C2v, C2' through the largest number of atoms for Dnh...) and symmetrized, since the 3D structures of PubChem or RDKit are not exactly symmetric.

A message summarizes this (`verbose=False` to hide it, e.g. in a loop).

### From a SMILES

In [ ]:
water = MolSym.from_molView(molView("O", source="smiles", display_now=False))
print(water.group)
water.orientation()
water.show()                 # the molecule alone, in its standard orientation

In [ ]:
nh3 = MolSym.from_molView(molView("N", source="smiles", display_now=False))
ch4 = MolSym.from_molView(molView("C", source="smiles", display_now=False))
benzene = MolSym.from_molView(molView("c1ccccc1", source="smiles", display_now=False))

In [ ]:
for smi in ["C=C", "C=O", "c1ccncc1", r"F/C=C\F", r"F/C=C/F", "ClC(Cl)Cl", "ClC(Cl)(Cl)Cl", "FB(F)F"]:
    m = MolSym.from_molView(molView(smi, source="smiles", display_now=False), verbose=False)
    print(f"{smi:15s} {m.group}")

### From an xyz file

The SMILES → 3D step requires a geometry optimization, which RDKit cannot always do: no force field for hypervalent compounds (SF6, PF5...) or organometallic compounds (ferrocene...). Idealized xyz models are provided in `data_examples/Symmetry/`.

In [ ]:
sf6 = MolSym.from_molView(molView(get_ppct_data("Symmetry/SF6_model.xyz"), source="file", display_now=False))
sf6.reduce()

fc_ecl = MolSym.from_molView(molView(get_ppct_data("Symmetry/ferrocene_eclipsed_model.xyz"), source="file", display_now=False))
fc_stag = MolSym.from_molView(molView(get_ppct_data("Symmetry/ferrocene_staggered_model.xyz"), source="file", display_now=False))

In [ ]:
# point group given: no detection by pymatgen
cerocene = MolSym.from_molView(molView(get_ppct_data("Symmetry/cerocene_model.xyz"), source="file", display_now=False), "D8h")
feco5 = MolSym.from_molView(molView(get_ppct_data("Symmetry/FeCO5_model.xyz"), source="file", display_now=False), "D3h")

### Coordinates typed by hand

`MolSym(symbols, coords, group)` also accepts coordinates typed by hand (Å), which is mostly useful to see the effect of the orientation.

In [ ]:
# water in the xz plane: moved to the yz plane (standard orientation of C2v), without pymatgen
water_xz = MolSym(["O", "H", "H"],
                  [[0, 0, 0.117], [0.757, 0, -0.469], [-0.757, 0, -0.469]],
                  "C2v")
print(np.round(water_xz.coords, 3))

In [ ]:
# C2 along x: pymatgen is needed to find the symmetry operations
water_x = MolSym(["O", "H", "H"],
                 [[0.117, 0, 0], [-0.469, 0.757, 0], [-0.469, -0.757, 0]],
                 "C2v")
print(np.round(water_x.coords, 3))

`standard_orientation=False`: the coordinates are kept as they are. If they are not in a standard orientation of the group, a warning is issued. The initial (centred) coordinates are always kept in `coords_input`.

In [ ]:
water_bad = MolSym(["O", "H", "H"],
                   [[0.117, 0, 0], [-0.469, 0.757, 0], [-0.469, -0.757, 0]],
                   "C2v", standard_orientation=False)
water_xz_raw = MolSym(["O", "H", "H"],
                      [[0, 0, 0.117], [0.757, 0, -0.469], [-0.757, 0, -0.469]],
                      "C2v", standard_orientation=False)
water_xz_raw.orientation()

## 4. Atom tracking, Γ_atoms, atomic orbitals

**What is Γ_atoms?** The basis is made of one function per atom, e.g. an s orbital centred on each atom (or simply the atoms themselves). A symmetry operation permutes the atoms, so its matrix in this basis is a permutation matrix: the character is the number of atoms that stay in place.

Reducing Γ_atoms tells how the atoms combine into symmetry-adapted linear combinations (SALCs). Each set of equivalent atoms gives its own contribution, e.g. for H2O: O → A1, the two H → A1 + B2, hence Γ_atoms = 2A1 + B2. For CH4: C → A1, the four H 1s orbitals → A1 + T2, the basis of the LCAO molecular orbitals.

### Water (C2v)

In [ ]:
water.track("C2")
water.track("σv(xz)")
water.track("σv'(yz)")
print("atoms fixed by C2:", water.fixed_atoms("C2"))

In [ ]:
water.gamma_atoms()
water.reduce()

In [ ]:
water.character_table()     # table + row Γ_atoms

### Matrices of the operations of the group: `matrices()`

In [ ]:
water.matrices()                         # one representative per class

### Ammonia (C3v) and an operation that is not a symmetry operation

In [ ]:
nh3.gamma_atoms()
nh3.reduce()
nh3.track(Cn(4))    # any matrix: C4 is not a symmetry operation of NH3

In [ ]:
nh3.matrices("σv", all_operations=True)    # all the operations of a class

### Methane (Td), benzene (D6h), bis(cyclooctatetraenyl)cerium (D8h)

In [ ]:
ch4.gamma_atoms()
ch4.reduce()
benzene.gamma_atoms()
benzene.reduce()
cerocene.reduce()

### Basis of valence atomic orbitals: `gamma_orbitals()`

For each set of equivalent atoms and each shell, the characters and the reduction give the symmetries of the SALCs, i.e. the building blocks of a molecular orbital diagram.
For an operation R, an atom that stays in place contributes χ_l(R): 1 for s, the trace of R for p (p orbitals transform as x, y, z), χ_l(θ) = sin((2l+1)θ/2)/sin(θ/2) in general; an atom that moves contributes 0.

- `basis="valence"` (default): s for H, He; s, p for the main group elements; s, p, d for the d block; s, p, d, f for the f block
- or the same shells on all the atoms: `"s"`, `"pz"`, `"sp"`, `["s", "px", "py"]`
- or a dict, e.g. `{"C": "sp", "H": "s"}`
- π basis: `"pz"` = p orbitals perpendicular to the molecular plane xy (`"px"` for a planar C2v molecule, in the yz plane); `"pxy"` = p orbitals in the molecular plane (σ basis)

In [ ]:
water.gamma_orbitals()                    # valence basis

In [ ]:
water.gamma_orbitals("s")                 # the same shells on all the atoms
water.gamma_orbitals(["s", "px", "py"])   # list: px and py separately (allowed in C2v)

In [ ]:
ch4.gamma_orbitals()

In [ ]:
benzene.orientation()                             # molecular plane xy -> π orbitals = pz
benzene.gamma_orbitals({"C": "pz"})              # π system: A2u + B2g + E1g + E2u
benzene.gamma_orbitals({"C": "s pxy", "H": "s"})  # σ system

In [ ]:
print(benzene.equivalent_atoms())               # sets of symmetry-equivalent atoms

In [ ]:
fc_stag.gamma_orbitals({"Fe": "spd", "C": "pz"})   # metal valence orbitals and Cp π orbitals
cerocene.gamma_orbitals({"Ce": "spdf", "C": "pz"}) # Ce 4f, 5d, 6s, 6p and COT π orbitals

## 5. 3D visualization

Colors: C2 red, C3 green, C4 blue, C5 orange, C6 purple, C8 brown; Sn axes dashed; planes in translucent blue; inversion center in black.

In [ ]:
nh3.show_elements(labels=True)     # all the elements

In [ ]:
ch4.show_elements("S4")            # a single class

In [ ]:
ch4.show_elements(["S4", "C2"])    # several classes (the label of the S4 axes is at the other end)

In [ ]:
benzene.show_elements("C2'")

In [ ]:
benzene.show_elements("σd")

In [ ]:
fc_stag.show_elements(["C5", "C2"])

In [ ]:
cerocene.show_elements("C8")

Before / after an operation: each atom keeps its label (the two views are linked).

In [ ]:
nh3.show_operation("C3")

In [ ]:
water.show_operation("σv(xz)", highlight=1)    # follow the atom H1

## 6. Vibrations

### Γ_3N and Γ_vib: `gamma_3N()`, `gamma_vib()`

- Γ_3N: basis of the 3N cartesian displacements of the atoms: χ_3N(R) = (number of atoms that stay in place) × χ_xyz(R), with χ_xyz(R) = trace of the matrix of R
- Γ_vib = Γ_3N − Γ_trans − Γ_rot, with χ_trans(R) = trace(R) (x, y, z) and χ_rot(R) = det(R) × trace(R) (Rx, Ry, Rz are axial vectors), followed by the IR / Raman activity of the modes

In [ ]:
water.gamma_3N()
water.gamma_vib()

In [ ]:
ch4.gamma_vib()        # A1 + E + 2T2: 2 IR bands (T2), 4 Raman bands

In [ ]:
benzene.gamma_vib()    # 20 modes: 4 IR bands, 7 Raman bands, no common band (centrosymmetric molecule)

`reduce(chi)` and `activity(chi)` accept any characters, e.g. those returned with `show=False`:

In [ ]:
chi = ch4.gamma_vib(show=False)
print(chi)
ch4.reduce(chi)
ch4.activity(chi)

### Basis of bond stretchings: `gamma_bonds()`

Internal coordinates: for an operation R, a bond sent onto itself (possibly with its two ends exchanged) contributes 1, a bond that moves contributes 0.
The reduction gives the symmetries of the stretching modes, for each set of equivalent bonds, followed by their IR / Raman activity.

- `bonds=None` (default): all the bonds, detected from the distances (`bonds()`, covalent radii)
- `bonds=["C-H"]`: bonds between given elements
- `bonds=[(0, 1), (0, 2)]`: explicit bonds (all the symmetry-equivalent bonds must be included)

In [ ]:
water.gamma_bonds()
ch4.gamma_bonds()

In [ ]:
benzene.gamma_bonds()              # ring C–C and C–H stretchings

In [ ]:
sf6.gamma_bonds()                  # IR: T1u; Raman: A1g + Eg (mutual exclusion: centrosymmetric molecule)

Number of ν(CO) bands of a metal carbonyl: Fe(CO)5 (D3h, idealized geometry).

In [ ]:
feco5.gamma_bonds(["C-O"])         # 2A1' + E' + A2'': 2 IR bands (E', A2'')
feco5.gamma_vib()                  # all the 27 vibrational modes